## NB1｜NumPy 從零實作：前向傳播

**前置理論：** [`theory/01a`](../theory/01a-prerequisites-intuition.md) 或 [`01b`](../theory/01b-prerequisites-math.md) → [`02`](../theory/02-attention-intuition.md) → [`03a`](../theory/03a-transformer-architecture.md)

**本 Notebook 目標：**
- 用 NumPy（不用 PyTorch）從零實作完整 Transformer 語言模型的前向傳播
- 理解每個元件（Embedding、Positional Encoding、Self-Attention、FFN、Block）的形狀與計算
- 訓練一個能生成文字的最小 LLM

**注意：** 本版本的反向傳播用 PyTorch autograd 概念做說明，不手刻梯度。完整手刻版請見 NB3。

---

# 用 NumPy 從零實作簡單大語言模型（LLM）

本筆記本展示一個最小可運行的 Transformer 語言模型，**使用 NumPy 進行矩陣運算**，取代純 Python 的巢狀迴圈。

架構包含：
1. Token Embedding
2. Positional Encoding
3. Self-Attention
4. Feed Forward Network
5. 訓練迴圈（Cross-Entropy Loss + 梯度下降）
6. 文字生成

**與純 Python 版本的差異：**
- 矩陣乘法 `@`、轉置 `.T`、廣播運算全部改用 NumPy
- 參數儲存為 `np.ndarray`，不再是 Python list of list
- softmax、relu、layer_norm 改用向量化寫法
- 速度顯著提升（特別在較長序列與較大模型）

## 1. 超參數與詞彙表設定

In [1]:
import math
import random
import numpy as np

# 固定亂數種子（NumPy + Python 兩者）
SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)  # 使用新式 Generator API

# 字元級詞彙表
VOCAB = list("abcdefghijklmnopqrstuvwxyz .,!?\n")
V = len(VOCAB)  # 詞彙表大小
D = 32  # d_model：隱藏層維度
D_FF = 64  # Feed Forward 中間層維度
H = 2  # 注意力頭數（目前單頭，供擴充參考）
SEQ = 16  # 最大序列長度
LR = 0.005  # 學習率

# 字元 ↔ ID 對照表
char2id = {c: i for i, c in enumerate(VOCAB)}
id2char = {i: c for c, i in char2id.items()}

print(f"詞彙表大小: {V}")
print(f"字元範例: {VOCAB[:10]}")

詞彙表大小: 32
字元範例: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']


## 2. 向量化基礎運算工具

用 NumPy 取代純 Python 的 matmul、softmax、ReLU 等基本運算。  
所有矩陣皆以 `np.ndarray` 表示，運算直接作用在整個陣列上（向量化）。

In [2]:
def randn(shape, scale=0.1):
    """常態分佈隨機初始化，回傳 np.ndarray"""
    return rng.standard_normal(shape) * scale


def softmax(x, axis=-1):
    """
    數值穩定的 softmax（向量化）
    x: np.ndarray，可為 1D (V,) 或 2D (T, V)
    """
    x = np.asarray(x, dtype=np.float64)
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def relu(x):
    """ReLU 激活函數（向量化）"""
    return np.maximum(0.0, x)


def layer_norm(x, eps=1e-6):
    """
    Layer Normalization（向量化）
    x: np.ndarray，shape (..., D)；對最後一軸正規化
    """
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)


# ── 快速功能測試 ──────────────────────────────
print("工具函數定義完成 ✓")

A = np.array([[1, 2], [3, 4]], dtype=float)
B = np.array([[5, 6], [7, 8]], dtype=float)
print("matmul test (A @ B):", (A @ B).tolist())  # [[19,22],[43,50]]
print("softmax test:       ", softmax(np.array([1.0, 2.0, 3.0])).round(3).tolist())
print(
    "layer_norm test:    ",
    layer_norm(np.array([[1.0, 2.0, 3.0, 4.0]])).round(3).tolist(),
)

工具函數定義完成 ✓
matmul test (A @ B): [[19.0, 22.0], [43.0, 50.0]]
softmax test:        [0.09, 0.245, 0.665]
layer_norm test:     [[-1.342, -0.447, 0.447, 1.342]]


## 3. Token Embedding 層

將每個 Token ID 映射到一個 D 維向量。  
權重矩陣 `W` 為 `np.ndarray(vocab_size, d_model)`，查表只需 `W[ids]` 一行。

In [3]:
class Embedding:
    """
    查找表：Token ID → D 維向量
    W shape: (vocab_size, d_model)
    """

    def __init__(self, vocab_size, d_model):
        self.W = randn((vocab_size, d_model), scale=0.02)  # (V, D)

    def forward(self, ids):
        """ids: list[int] or 1D array → ndarray (T, D)"""
        return self.W[ids]  # NumPy fancy indexing，一行搞定

    def params(self):
        return [self.W]


# 示範
emb = Embedding(V, D)
test_ids = [char2id["h"], char2id["i"]]
vecs = emb.forward(test_ids)
print(f"輸入 IDs: {test_ids}  →  輸出向量形狀: {vecs.shape}")
print(f"第一個向量（前 5 維）: {vecs[0, :5].round(4).tolist()}")

輸入 IDs: [7, 8]  →  輸出向量形狀: (2, 32)
第一個向量（前 5 維）: [0.0018, 0.0046, 0.0503, 0.0375, -0.0171]


## 4. Positional Encoding

用 sin/cos 函數為每個位置加入位置資訊。  
NumPy 版用廣播一次算出整個 PE 矩陣，不需雙層迴圈。

$$PE[pos, 2i] = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad
PE[pos, 2i+1] = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

In [4]:
def positional_encoding(seq_len, d_model):
    """
    標準 Transformer Positional Encoding（向量化）
    回傳 np.ndarray shape (seq_len, d_model)
    """
    pos = np.arange(seq_len)[:, np.newaxis]  # (T, 1)
    i = np.arange(0, d_model, 2)[np.newaxis, :]  # (1, D/2)
    angle = pos / (10000 ** (i / d_model))  # (T, D/2)  廣播

    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle)  # 偶數維度
    pe[:, 1::2] = np.cos(angle[:, : d_model // 2])  # 奇數維度（長度對齊）
    return pe


def add_positional(X):
    """X: ndarray (T, D) → 加入 PE 後的 (T, D)"""
    pe = positional_encoding(X.shape[0], X.shape[1])
    return X + pe  # NumPy 廣播加法


# 示範
pe = positional_encoding(SEQ, D)
print(f"Positional Encoding 形狀: {pe.shape}")
print(f"位置 0 的前 6 維: {pe[0, :6].round(4).tolist()}")
print(f"位置 1 的前 6 維: {pe[1, :6].round(4).tolist()}")

Positional Encoding 形狀: (16, 32)
位置 0 的前 6 維: [0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
位置 1 的前 6 維: [0.8415, 0.5403, 0.5332, 0.846, 0.311, 0.9504]


## 5. Self-Attention 層

核心公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

NumPy 版可一次算出整個 `(T, T)` 注意力矩陣，不需逐行迴圈。  
Causal mask 用 `np.triu` 一行產生。

In [5]:
class SelfAttention:
    """
    單頭 Self-Attention（NumPy 向量化版）
    參數：Wq, Wk, Wv, Wo  各為 ndarray (D, D)
    """

    def __init__(self, d_model):
        scale = 0.02
        self.Wq = randn((d_model, d_model), scale)
        self.Wk = randn((d_model, d_model), scale)
        self.Wv = randn((d_model, d_model), scale)
        self.Wo = randn((d_model, d_model), scale)
        self.d_model = d_model

    def forward(self, X, mask=True):
        """
        X: ndarray (T, D)
        mask=True 表示 causal masking（只看過去）
        回傳: ndarray (T, D)
        """
        T, dk = X.shape[0], math.sqrt(self.d_model)

        Q = X @ self.Wq  # (T, D)
        K = X @ self.Wk  # (T, D)
        V = X @ self.Wv  # (T, D)

        # 注意力分數 (T, T)
        scores = (Q @ K.T) / dk

        # Causal mask：上三角（不含對角線）設為 -inf
        if mask:
            causal = np.triu(np.full((T, T), -1e9), k=1)
            scores = scores + causal

        # Softmax（對每一列）
        attn = softmax(scores, axis=-1)  # (T, T)

        # 加權求和 V → 輸出投影
        out = (attn @ V) @ self.Wo  # (T, D)
        return out

    def params(self):
        return [self.Wq, self.Wk, self.Wv, self.Wo]


# 示範
attn_layer = SelfAttention(D)
dummy_input = pe[:5]  # (5, D)
out = attn_layer.forward(dummy_input)
print(f"Self-Attention 輸出形狀: {out.shape}")
print(f"第一個輸出向量（前 5 維）: {out[0, :5].round(4).tolist()}")

Self-Attention 輸出形狀: (5, 32)
第一個輸出向量（前 5 維）: [0.0166, -0.0037, 0.0049, 0.0063, 0.0002]


## 6. Feed Forward Network

兩層線性變換 + ReLU：
$$\text{FFN}(X) = \text{ReLU}(XW_1 + b_1)W_2 + b_2$$

NumPy 版直接對整個序列矩陣 `(T, D)` 做矩陣乘法，不需對每個位置迴圈。

In [6]:
class FeedForward:
    """
    Position-wise Feed Forward Network（NumPy 向量化版）
    同時處理所有 T 個位置，不需逐位置迴圈
    """

    def __init__(self, d_model, d_ff):
        self.W1 = randn((d_model, d_ff), 0.02)  # (D, D_FF)
        self.b1 = np.zeros(d_ff)  # (D_FF,)
        self.W2 = randn((d_ff, d_model), 0.02)  # (D_FF, D)
        self.b2 = np.zeros(d_model)  # (D,)

    def forward(self, X):
        """
        X: ndarray (T, D) → 回傳 (T, D)
        整個序列一次前向傳播（向量化）
        """
        h = relu(X @ self.W1 + self.b1)  # (T, D_FF)
        return h @ self.W2 + self.b2  # (T, D)

    def params(self):
        return [self.W1, self.W2]


print("FeedForward 定義完成 ✓")

# 快速測試
ff = FeedForward(D, D_FF)
dummy = randn((5, D))
ff_out = ff.forward(dummy)
print(f"FeedForward 輸出形狀: {ff_out.shape}")  # (5, 32)

FeedForward 定義完成 ✓
FeedForward 輸出形狀: (5, 32)


## 7. Transformer Block

組合 Self-Attention + Feed Forward + 殘差連接 + Layer Norm。  
殘差連接直接用 `+` 運算子，Layer Norm 對 `(T, D)` 矩陣整批正規化。

In [7]:
class TransformerBlock:
    """
    單個 Transformer 層（NumPy 向量化版）：
      X = X + SelfAttention(LayerNorm(X))   # 殘差 + 注意力
      X = X + FFN(LayerNorm(X))             # 殘差 + FFN
    """

    def __init__(self, d_model, d_ff):
        self.attn = SelfAttention(d_model)
        self.ff = FeedForward(d_model, d_ff)

    def forward(self, X):
        """
        X: ndarray (T, D) → 回傳 (T, D)
        layer_norm 對整個矩陣批次操作（沿最後一軸）
        """
        # Self-Attention 子層 + 殘差
        X = X + self.attn.forward(layer_norm(X))
        # Feed Forward 子層 + 殘差
        X = X + self.ff.forward(layer_norm(X))
        return X

    def params(self):
        return self.attn.params() + self.ff.params()


print("TransformerBlock 定義完成 ✓")

# 快速測試
block = TransformerBlock(D, D_FF)
blk_out = block.forward(dummy)
print(f"TransformerBlock 輸出形狀: {blk_out.shape}")  # (5, 32)

TransformerBlock 定義完成 ✓
TransformerBlock 輸出形狀: (5, 32)


## 8. 完整語言模型

整合所有組件，加上輸出投影層（`(T, D) → (T, V)` logits）。  
輸出層用 `X @ Wout` 一行完成，不需對每個位置迴圈。

In [8]:
class MiniLLM:
    """
    最小可運行 Transformer 語言模型（NumPy 版）
    架構：Embedding → +PE → TransformerBlock × N → LayerNorm → Linear
    所有參數皆為 np.ndarray，前向傳播全程向量化
    """

    def __init__(self, vocab_size, d_model, d_ff, n_layers=1):
        self.emb = Embedding(vocab_size, d_model)
        self.blocks = [TransformerBlock(d_model, d_ff) for _ in range(n_layers)]
        self.Wout = randn((d_model, vocab_size), 0.02)  # (D, V)
        self.d_model = d_model
        self.vocab_size = vocab_size

    def forward(self, ids):
        """
        ids: list[int] or 1D array
        回傳: ndarray (T, vocab_size)  每個位置的 logits
        """
        X = self.emb.forward(ids)  # (T, D)
        X = add_positional(X)  # (T, D) + PE
        for block in self.blocks:
            X = block.forward(X)  # (T, D)
        X = layer_norm(X)  # 最終 LayerNorm
        logits = X @ self.Wout  # (T, V)  ← 純矩陣乘法
        return logits

    def all_params(self):
        """取得所有可訓練參數（ndarray 列表）"""
        p = self.emb.params() + [self.Wout]
        for b in self.blocks:
            p += b.params()
        return p


# 建立模型
rng = np.random.default_rng(SEED)  # 重設種子以利重現
model = MiniLLM(vocab_size=V, d_model=D, d_ff=D_FF, n_layers=1)

# 快速前向傳播測試
test_text = "hello"
test_ids = [char2id.get(c, 0) for c in test_text]
logits = model.forward(test_ids)
print(f"輸入: '{test_text}'  ({len(test_ids)} tokens)")
print(f"輸出 logits 形狀: {logits.shape}")
probs = softmax(logits[-1])
top_id = int(np.argmax(probs))
print(f"預測下一個字元: '{id2char[top_id]}'（未訓練，隨機）")

輸入: 'hello'  (5 tokens)
輸出 logits 形狀: (5, 32)
預測下一個字元: 's'（未訓練，隨機）


## 9. 損失函數與訓練

使用 **Cross-Entropy Loss** 訓練模型預測下一個 Token。  
NumPy 版用進階索引 `probs[np.arange(T), targets]` 同時取出所有目標位置的機率，不需逐 token 迴圈。

梯度用數值微分（有限差分法）計算，僅用於教學示範。

In [9]:
def cross_entropy_loss(logits, target_ids):
    """
    向量化 Cross-Entropy Loss
    logits:     ndarray (T, V)
    target_ids: list[int] or ndarray (T,)
    """
    probs = softmax(logits, axis=-1)  # (T, V)
    targets = np.asarray(target_ids)  # (T,)
    # 進階索引：每列取出對應目標字元的機率
    correct = probs[np.arange(len(targets)), targets]  # (T,)
    return -np.mean(np.log(correct + 1e-9))


def compute_loss(model, text):
    """對一段文字計算語言模型損失"""
    ids = [char2id.get(c, 0) for c in text]
    if len(ids) < 2:
        return 0.0
    input_ids = ids[:-1]  # 輸入：去掉最後一個
    target_ids = ids[1:]  # 目標：去掉第一個（預測下一個）
    logits = model.forward(input_ids)  # (T-1, V)
    return float(cross_entropy_loss(logits, target_ids))


def numerical_gradient(model, text, eps=1e-3):
    """
    數值微分（有限差分法）計算所有參數的梯度
    grad ≈ (f(x+ε) - f(x-ε)) / (2ε)
    注意：這很慢，僅用於教學示範
    """
    grads = []
    for param in model.all_params():
        grad = np.zeros_like(param)
        # np.ndindex 遍歷任意形狀陣列的所有索引
        for idx in np.ndindex(param.shape):
            orig = param[idx]
            param[idx] = orig + eps
            lp = compute_loss(model, text)
            param[idx] = orig - eps
            lm = compute_loss(model, text)
            param[idx] = orig
            grad[idx] = (lp - lm) / (2 * eps)
        grads.append(grad)
    return grads


def update_params(model, grads, lr):
    """梯度下降更新所有參數（就地修改）"""
    for param, grad in zip(model.all_params(), grads):
        param -= lr * grad  # NumPy 就地運算


# 測試損失計算
loss = compute_loss(model, "hello")
print(f"未訓練時的損失: {loss:.4f}")
print(f"（隨機猜測的理論損失: {math.log(V):.4f}）")

未訓練時的損失: 3.5113
（隨機猜測的理論損失: 3.4657）


## 10. 訓練迴圈

⚠️ **注意**：數值微分仍然很慢（O(參數數量) 次前向傳播），這裡用少量迭代示範訓練過程。  
但每次前向傳播本身已因向量化而大幅加速。

本範例只更新 **Embedding 矩陣**（其餘參數固定），因此損失下降幅度極小——這是預期行為，不是 bug。  
完整更新所有參數需對幾千個元素各做兩次前向傳播，在教學環境中需數分鐘。  
若要看完整訓練效果（損失明顯下降），請移至 **NB2**（PyTorch 版本，使用 autograd 自動計算梯度）。

實際生產環境會用反向傳播（autograd）來計算梯度。


In [10]:
train_text = "hello world"

print(f"訓練文字: '{train_text}'")
print(f"字元數: {len(train_text)}")
print("\n開始訓練（僅對 Embedding 做數值微分示範）...\n")

# 重新初始化較小模型以加速示範
rng = np.random.default_rng(SEED)
D_SMALL, DFF_SMALL = 16, 32
model_small = MiniLLM(vocab_size=V, d_model=D_SMALL, d_ff=DFF_SMALL, n_layers=1)

N_STEPS = 10
losses = []
eps = 1e-3

for step in range(N_STEPS):
    loss = compute_loss(model_small, train_text)
    losses.append(loss)

    # 僅對 Embedding 矩陣計算梯度（加速示範）
    W = model_small.emb.W  # ndarray (V, D_SMALL)
    grad = np.zeros_like(W)

    for idx in np.ndindex(W.shape):
        orig = W[idx]
        W[idx] = orig + eps
        lp = compute_loss(model_small, train_text)
        W[idx] = orig - eps
        lm = compute_loss(model_small, train_text)
        W[idx] = orig
        grad[idx] = (lp - lm) / (2 * eps)

    W -= LR * grad  # 就地更新

    print(f"Step {step+1:3d}/{N_STEPS}  Loss: {loss:.4f}")

print(f"\n訓練完成！損失從 {losses[0]:.4f} 降至 {losses[-1]:.4f}")

訓練文字: 'hello world'
字元數: 11

開始訓練（僅對 Embedding 做數值微分示範）...

Step   1/10  Loss: 3.4389


Step   2/10  Loss: 3.4389


Step   3/10  Loss: 3.4389
Step   4/10  Loss: 3.4389


Step   5/10  Loss: 3.4388


Step   6/10  Loss: 3.4388
Step   7/10  Loss: 3.4388


Step   8/10  Loss: 3.4388


Step   9/10  Loss: 3.4388
Step  10/10  Loss: 3.4388

訓練完成！損失從 3.4389 降至 3.4388


## 11. 文字生成

訓練完成後，用模型「補全」文字（Autoregressive 生成）。  
機率採樣改用 `np.random.choice`，更簡潔且語義清晰。

In [11]:
def generate(model, prompt, max_new_tokens=20, temperature=1.0):
    """
    自回歸生成文字
    temperature > 1 → 更隨機；temperature < 1 → 更確定
    使用 np.random.choice 進行機率採樣
    """
    ids = [char2id.get(c, 0) for c in prompt]
    generated = list(prompt)

    for _ in range(max_new_tokens):
        context = ids[-SEQ:]  # 滑動視窗
        logits = model.forward(context)  # (T, V)
        last_logits = logits[-1] / temperature  # (V,)  temperature scaling
        probs = softmax(last_logits)  # (V,)
        next_id = int(np.random.choice(V, p=probs))  # 機率採樣
        ids.append(next_id)
        generated.append(id2char[next_id])

    return "".join(generated)


# 生成示範
np.random.seed(0)
prompt = "h"
print(f"Prompt: '{prompt}'")

for temp, label in [(0.5, "低溫（保守）"), (1.0, "標準"), (2.0, "高溫（創意）")]:
    result = generate(model_small, prompt, max_new_tokens=15, temperature=temp)
    print(f"  temperature={temp} ({label}): '{result}'")

print("\n（模型訓練步數極少，生成結果仍偏隨機；增加訓練步數可改善）")

Prompt: 'h'
  temperature=0.5 (低溫（保守）): 'hrwtrnun,?lyqr!c'
  temperature=1.0 (標準): 'hca y.
zoydue?qn'
  temperature=2.0 (高溫（創意）): 'hiyosattt?vlnwbv'

（模型訓練步數極少，生成結果仍偏隨機；增加訓練步數可改善）


## 12. 整合：端對端示範

In [12]:
# 查看損失下降趨勢
print("=" * 40)
print("損失下降紀錄")
print("=" * 40)
for i, l in enumerate(losses):
    bar = "█" * int(l * 5)
    print(f"Step {i+1:2d}: {l:.4f}  {bar}")

print("\n" + "=" * 40)
print("模型參數量統計")
print("=" * 40)
total = sum(p.size for p in model_small.all_params())  # ndarray.size 直接取元素數
print(f"總參數量: {total:,}")
print(f"（GPT-2 有 1.17 億參數；GPT-4 估計超過 1 兆）")

print("\n" + "=" * 40)
print("NumPy 參數形狀一覽")
print("=" * 40)
names = [
    "Embedding.W",
    "Wout",
    "Attn.Wq",
    "Attn.Wk",
    "Attn.Wv",
    "Attn.Wo",
    "FFN.W1",
    "FFN.W2",
]
for name, p in zip(names, model_small.all_params()):
    print(f"  {name:15s}  shape={str(p.shape):15s}  dtype={p.dtype}")

損失下降紀錄
Step  1: 3.4389  █████████████████
Step  2: 3.4389  █████████████████
Step  3: 3.4389  █████████████████
Step  4: 3.4389  █████████████████
Step  5: 3.4388  █████████████████
Step  6: 3.4388  █████████████████
Step  7: 3.4388  █████████████████
Step  8: 3.4388  █████████████████
Step  9: 3.4388  █████████████████
Step 10: 3.4388  █████████████████

模型參數量統計
總參數量: 3,072
（GPT-2 有 1.17 億參數；GPT-4 估計超過 1 兆）

NumPy 參數形狀一覽
  Embedding.W      shape=(32, 16)         dtype=float64
  Wout             shape=(16, 32)         dtype=float64
  Attn.Wq          shape=(16, 16)         dtype=float64
  Attn.Wk          shape=(16, 16)         dtype=float64
  Attn.Wv          shape=(16, 16)         dtype=float64
  Attn.Wo          shape=(16, 16)         dtype=float64
  FFN.W1           shape=(16, 32)         dtype=float64
  FFN.W2           shape=(32, 16)         dtype=float64


## 13. 與 03a 對照：逐步數值驗證

[`theory/03b-transformer-architecture-example.md`](../theory/03b-transformer-architecture-example.md) 用一組 $2\times4$ 的輸入，把 **Pre-LN → Multi-Head Attention → 殘差 → FFN → 殘差** 從頭手算到尾。

下方 cell **自成一體**（不依賴本筆記本前面的類別），硬編碼 03a 的輸入 $X$ 與所有權重，逐步印出 $\text{LN}(X)\to S\to E\to A\to C\to O\to Z'\to \text{LN}(Z')\to F\to Y$，以及 Positional Encoding 表與「相對位置 = 旋轉」驗證。每個輸出都可對著 03a 逐格核對。

> 採與 03a 一致的慣例：Pre-LN、row-vector、投影矩陣 $W\in\mathbb{R}^{d\times d_k}$、LayerNorm 用 population variance（$\gamma=1,\beta=0$）。

In [13]:
# =====================================================================
# §13  與 theory/03b-transformer-architecture-example.md 對照：逐步數值驗證
# ---------------------------------------------------------------------
# 本 cell 自成一體（不依賴上面的類別），硬編碼 03a 的 X 與所有權重，
# 逐步印出 LN(X) → S → E → A → C → O → Z' → LN(Z') → F → Y 與 PE 表，
# 方便對著 03a 逐格核對。採 Pre-LN、row-vector、W∈R^{d×d_k} 慣例。
# =====================================================================
import numpy as np
np.set_printoptions(precision=4, suppress=True)


def softmax_row(M):
    M = M - M.max(axis=1, keepdims=True)
    e = np.exp(M)
    return e / e.sum(axis=1, keepdims=True)


def ln(X, gamma, beta, eps=0.0):          # LayerNorm，對每列（最後一軸）正規化
    mu = X.mean(axis=1, keepdims=True)
    var = X.var(axis=1, keepdims=True)    # population variance，與 03a 手算一致
    return gamma * (X - mu) / np.sqrt(var + eps) + beta


# ---- 設定（= 03a §0）----
X = np.array([[1., 0, 0, 1], [0, 1, 1, 0]])
dk = 2
WQ1 = np.array([[1., 0], [0, 1], [0, 0], [0, 0]])    # Head 1：取前 2 維
WQ2 = np.array([[0., 0], [0, 0], [1, 0], [0, 1]])    # Head 2：取後 2 維
WO = 0.5 * np.array([[1., 0, 1, 0], [0, 1, 0, 1],
                     [1, 0, -1, 0], [0, 1, 0, -1]])
g, b = np.ones(4), np.zeros(4)                        # LayerNorm γ, β（初始值）
W1 = 0.5 * np.array([[1, -1, 0, 1, 1, 0, -1, 0],
                     [0, 1, 1, -1, 0, 1, 0, -1],
                     [1, 0, -1, 0, 1, -1, 0, 1],
                     [-1, 1, 0, 1, 0, 0, 1, -1]], float)
W2 = 0.5 * np.array([[1, 0, 1, -1], [0, 1, -1, 0], [1, -1, 0, 1], [-1, 0, 1, 0],
                     [0, 1, 0, 1], [1, 0, -1, 0], [0, -1, 0, 1], [-1, 0, 1, 0]], float)

# ---- §1 Pre-LN ----
Xn = ln(X, g, b)
print("§1  LN(X) =\n", Xn)


# ---- §2-§3 各 head（本例 WQ=WK=WV）----
def head(Xin, W):
    Q = K = V = Xin @ W
    S = Q @ K.T            # 原始分數
    E = S / np.sqrt(dk)    # 縮放後分數
    A = softmax_row(E)     # 注意力權重
    return S, E, A, A @ V  # 末項為 context C


S1, E1, A1, C1 = head(Xn, WQ1)
S2, E2, A2, C2 = head(Xn, WQ2)
print("\n§2  Head1  S =\n", S1, "\n     E =\n", E1, "\n     A =\n", A1, "\n     C =\n", C1)
print("\n§3  Head2  A =\n", A2, "\n     C =\n", C2)

O = np.concatenate([C1, C2], axis=1) @ WO
print("\n§3  O = Concat·WO =\n", O)

# ---- §4 第一殘差（Pre-LN：加在原始 X 上）----
Zp = X + O
print("\n§4  Z' = X + O =\n", Zp)

# ---- §5 第二子層：LN → FFN → 第二殘差 ----
Zn = ln(Zp, g, b)
F = np.maximum(Zn @ W1, 0) @ W2          # ReLU 為唯一非線性
Y = Zp + F
print("\n§5  LN(Z') =\n", Zn, "\n     F =\n", F, "\n     Y = Z'+F (Block 輸出) =\n", Y)


# ---- §6 Positional Encoding 與「相對位置 = 旋轉」驗證 ----
def pe(T, d):
    P = np.zeros((T, d))
    for i in range(T):
        for m in range(d // 2):
            w = 1 / 10000 ** (2 * m / d)
            P[i, 2 * m], P[i, 2 * m + 1] = np.sin(w * i), np.cos(w * i)
    return P


P = pe(4, 4)
print("\n§6  PE(pos 0..3, d=4) =\n", P)

# 驗證 p_{i+δ} = R(ω_m·δ)·p_i  （取 m=0, i=1, δ=2）
w, i, delta = 1.0, 1, 2
R = np.array([[np.cos(w * delta), np.sin(w * delta)],
              [-np.sin(w * delta), np.cos(w * delta)]])
lhs, rhs = P[i + delta, 0:2], R @ P[i, 0:2]
print("§6  旋轉驗證  lhs =", lhs.round(4), " rhs =", rhs.round(4),
      " 相符:", np.allclose(lhs, rhs))

§1  LN(X) =
 [[ 1. -1. -1.  1.]
 [-1.  1.  1. -1.]]

§2  Head1  S =
 [[ 2. -2.]
 [-2.  2.]] 
     E =
 [[ 1.4142 -1.4142]
 [-1.4142  1.4142]] 
     A =
 [[0.9442 0.0558]
 [0.0558 0.9442]] 
     C =
 [[ 0.8884 -0.8884]
 [-0.8884  0.8884]]

§3  Head2  A =
 [[0.9442 0.0558]
 [0.0558 0.9442]] 
     C =
 [[-0.8884  0.8884]
 [ 0.8884 -0.8884]]

§3  O = Concat·WO =
 [[ 0.      0.      0.8884 -0.8884]
 [ 0.      0.     -0.8884  0.8884]]

§4  Z' = X + O =
 [[1.     0.     0.8884 0.1116]
 [0.     1.     0.1116 0.8884]]

§5  LN(Z') =
 [[ 1.1169 -1.1169  0.8675 -0.8675]
 [-1.1169  1.1169 -0.8675  0.8675]] 
     F =
 [[-0.3415  0.4961  1.7675 -0.2169]
 [ 0.9922 -0.2169 -1.2714  0.9922]] 
     Y = Z'+F (Block 輸出) =
 [[ 0.6585  0.4961  2.6559 -0.1053]
 [ 0.9922  0.7831 -1.1598  1.8806]]

§6  PE(pos 0..3, d=4) =
 [[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]
 [ 0.1411 -0.99    0.03    0.9996]]
§6  旋轉驗證  lhs = [ 0.1411 -0.99  ]  rhs = [ 0.1411 

## 延伸學習

這個迷你 LLM 展示了核心概念，但真實 LLM 還有許多改進：

| 功能 | 純 Python 版 | **NumPy 版（本筆記本）** | 真實 LLM |
|------|------------|----------------------|----------|
| 矩陣運算 | 純 Python 迴圈 | **NumPy（向量化）** | CUDA GPU |
| 梯度計算 | 數值微分（慢）| 數值微分（慢）| 反向傳播（autograd）|
| 詞彙表 | 字元級（32）| 字元級（32）| BPE（50,000+）|
| 參數量 | ~幾千 | ~幾千 | 數十億 |
| 注意力 | 單頭 | 單頭 | 多頭（Multi-Head）|
| 訓練資料 | 幾個詞 | 幾個詞 | 萬億 Token |

**NumPy 版的主要改進點回顧：**
- `W[ids]`：Embedding 查表改為 NumPy fancy indexing
- `Q @ K.T`：注意力分數改為矩陣乘法（無迴圈）
- `np.triu`：Causal mask 一行產生
- `softmax(axis=-1)`：一次對整個 `(T, V)` 矩陣套用
- `probs[np.arange(T), targets]`：Cross-entropy 進階索引
- `np.random.choice`：機率採樣更直觀
- `p.size`：參數計數更簡潔

**推薦下一步：**
- 🔗 [Andrej Karpathy 的 nanoGPT](https://github.com/karpathy/nanoGPT)（真正的小型 GPT）
- 📖 Attention Is All You Need（原始 Transformer 論文）
- 🛠️ 用 PyTorch 實作自動微分版本（梯度計算自動化）